# [9.2] Chain-of-Thought Faithfulness - Exercises

Written chain-of-thought is evidence about what a model said, not direct evidence about what caused the answer. In this notebook you build a small faithfulness harness: read out hidden answers before the final token, compare against visible answers, test whether a hidden-state movement changes answer logits, and reject shortcuts with text-only and label-shuffle controls.

<img src="../../instructions/assets/cot_faithfulness_validation_loop.svg" width="860">

By the end, you should be able to explain why a probe result is only correlational, why the LM-head readout patch is a narrow causal check, why the text-only baseline matters, and why the final report does not claim a broad CoT faithfulness benchmark.


In [ ]:
from dataclasses import dataclass
import json
import sys
from pathlib import Path
from typing import Literal

import torch as t

chapter = "chapter9_alignment_interpretability"
section = "part2_cot_faithfulness"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_cot_faithfulness.tests as tests
import part2_cot_faithfulness.utils as utils

MAIN = __name__ == "__main__"
CoTCondition = Literal["no_cot", "faithful_cot", "biased_cot", "posthoc"]

GT_TIER = "GT-3"
EXERCISE_ID = "9_2_chain_of_thought_faithfulness"
EXPECTED_RUNTIME = "30-45 minutes for exercises; a few minutes for the pinned Pythia CUDA preflight"
REQUIRES_GPU = True


## Report Types and Guard Helpers

The dataclasses keep the notebook outputs stable enough for tests and verification reports. The finite-value guards are not decoration: a NaN-filled probe can otherwise produce arbitrary argmaxes and still look like a number.


In [ ]:
@dataclass(frozen=True)
class PreFinalAnswerProbeReport:
    hidden_answer_accuracy: float
    final_answer_agreement: float
    predicts_hidden_answer: bool


@dataclass(frozen=True)
class HiddenAnswerPatchingReport:
    original_answer: int
    patched_answer: int
    changed_output: bool


@dataclass(frozen=True)
class CoTTextBaselineReport:
    detector_recall: float
    text_only_recall: float
    text_only_misses_cases: bool


@dataclass(frozen=True)
class FeatureDetectorReport:
    feature_accuracy: float
    baseline_accuracy: float
    improves_detection: bool


@dataclass(frozen=True)
class CoTConditionComparisonReport:
    condition_accuracies: dict[str, float]
    biased_gap: float
    posthoc_gap: float


def _require_finite_tensor(name: str, tensor: t.Tensor) -> None:
    if tensor.numel() == 0:
        raise ValueError(f"{name} must be non-empty.")
    if not t.isfinite(tensor.float()).all():
        raise ValueError(f"{name} must contain only finite values.")


def _require_finite_scalar(name: str, value: float) -> None:
    value_tensor = t.tensor(value, dtype=t.float32)
    if not t.isfinite(value_tensor):
        raise ValueError(f"{name} must be finite.")


## Top-1 Answer Accuracy

> ```yaml
> Difficulty: easy
> Importance: high
> ```

Implement top-1 accuracy for answer logits. The target ids must match every leading dimension of `logits`.

<details>
<summary>Expected output</summary>

```text
All tests in `test_prediction_accuracy_checks_top1_predictions` passed!
```

The fixture has three correct predictions out of four, rejects a shape mismatch, and rejects non-finite logits.
</details>

<details>
<summary>Common bugs</summary>

- Comparing logits directly to labels instead of taking `argmax(dim=-1)`.
- Flattening before shape checks and hiding a batch-shape bug.
- Returning a rank-0 tensor instead of a Python float.
</details>

<details>
<summary>Help - why start here?</summary>

Every later report uses answer-token accuracy or argmax changes. If this helper accepts bad tensors or wrong axes, the rest of the notebook can produce convincing-looking nonsense.
</details>

<details>
<summary>Solution</summary>

Use `logits.argmax(dim=-1)`, compare with `target_token_ids`, average the boolean mask, and call the finite guards before scoring.
</details>


In [ ]:
def prediction_accuracy(logits: t.Tensor, target_token_ids: t.Tensor) -> float:
    raise NotImplementedError()


tests.test_prediction_accuracy_checks_top1_predictions(prediction_accuracy)


## Pre-final Hidden-Answer Probe

> ```yaml
> Difficulty: medium
> Importance: high
> ```

A pre-final probe can predict the hidden/private answer even when the final visible answer disagrees. Report both quantities.

<details>
<summary>Expected output</summary>

```text
All tests in `test_pre_final_answer_probe_report_predicts_hidden_answer` passed!
```

The toy probe has hidden-answer accuracy `1.0`, final-answer agreement `2/3`, and `predicts_hidden_answer == True`.
</details>

<details>
<summary>Common bugs</summary>

- Scoring the probe against final visible answers instead of hidden answers.
- Treating final-answer disagreement as a probe failure.
- Forgetting to reject empty probe batches.
</details>

<details>
<summary>Help - what does this prove?</summary>

It proves only that the hidden state contains linearly readable private-answer information in the toy fixture. It does not prove the model used that information causally.
</details>

<details>
<summary>Solution</summary>

Compute hidden-answer accuracy with `prediction_accuracy`, compute final-answer agreement from the same probe predictions, and threshold only the hidden-answer accuracy.
</details>


In [ ]:
def pre_final_answer_probe_report(
    probe_logits: t.Tensor,
    hidden_answer_ids: t.Tensor,
    final_answer_ids: t.Tensor,
    *,
    min_hidden_accuracy: float = 0.8,
) -> PreFinalAnswerProbeReport:
    raise NotImplementedError()


tests.test_pre_final_answer_probe_report_predicts_hidden_answer(
    pre_final_answer_probe_report,
)


## Hidden-Answer Readout Patch

> ```yaml
> Difficulty: medium
> Importance: high
> ```

Given original and patched answer-logit vectors, report whether the answer-token argmax changed.

<details>
<summary>Expected output</summary>

```text
All tests in `test_hidden_answer_patching_report_flags_answer_flip` passed!
```

The toy patch moves the answer from id `0` to id `1`.
</details>

<details>
<summary>Common bugs</summary>

- Using a logit-difference threshold instead of the answer argmax.
- Accepting rank-2 tensors when this report expects one logit vector before and after patching.
- Forgetting that non-finite logits make argmax meaningless.
</details>

<details>
<summary>Help - why is the claim narrow?</summary>

The report path moves a hidden vector and reads it through the LM head. That is evidence about answer-token readout, not a full hook-based activation patch through a forward pass.
</details>

<details>
<summary>Solution</summary>

Check rank and shape, call the finite guards, take both argmaxes, convert them to Python `int`s, and mark `changed_output` when they differ.
</details>


In [ ]:
def hidden_answer_patching_report(
    original_answer_logits: t.Tensor,
    patched_answer_logits: t.Tensor,
) -> HiddenAnswerPatchingReport:
    raise NotImplementedError()


tests.test_hidden_answer_patching_report_flags_answer_flip(
    hidden_answer_patching_report,
)


## Text-only Baseline

> ```yaml
> Difficulty: medium
> Importance: high
> ```

Compute recall on unfaithful cases for a white-box detector and for a visible-text-only baseline.

<details>
<summary>Expected output</summary>

```text
All tests in `test_cot_text_baseline_report_keeps_recall_gap` passed!
```

The detector recall is `1.0`, text-only recall is `0.5`, and `text_only_misses_cases == True`.
</details>

<details>
<summary>Common bugs</summary>

- Computing accuracy instead of recall over positive/unfaithful examples.
- Letting an all-negative batch divide by zero.
- Comparing raw float scores without converting to boolean predictions.
</details>

<details>
<summary>Help - why is text-only not enough?</summary>

A rationale string can contain obvious cues. The CUDA preflight uses a lexical post-hoc cue as the text baseline, and the hidden-state detector must catch cases that cue misses.
</details>

<details>
<summary>Solution</summary>

Flatten predictions and labels, require at least one positive label, compute true-positive recall, then compare detector and text-only recall.
</details>


In [ ]:
def _binary_recall(predictions: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def cot_text_baseline_report(
    detector_predictions: t.Tensor,
    text_only_predictions: t.Tensor,
    unfaithful_labels: t.Tensor,
) -> CoTTextBaselineReport:
    raise NotImplementedError()


tests.test_cot_text_baseline_report_keeps_recall_gap(cot_text_baseline_report)


## Feature-level Detector

> ```yaml
> Difficulty: medium
> Importance: high
> ```

Threshold an internal feature score and a baseline score under the same labels.

<details>
<summary>Expected output</summary>

```text
All tests in `test_feature_detector_report_scores_thresholded_predictions` passed!
```

The feature detector accuracy is `1.0`, the baseline accuracy is `0.75`, and `improves_detection == True`.
</details>

<details>
<summary>Common bugs</summary>

- Thresholding one score vector but not the other.
- Comparing raw score magnitudes to labels.
- Accepting NaN scores or a NaN threshold.
</details>

<details>
<summary>Help - what is a feature score here?</summary>

The toy notebook receives scores directly. In a real white-box monitor, the score might come from a probe, an SAE feature, or another internal readout. The baseline comparison is the transferable habit.
</details>

<details>
<summary>Solution</summary>

Apply the same `>= threshold` rule to both score vectors, compare against labels, and report both accuracies.
</details>


In [ ]:
def feature_detector_report(
    feature_scores: t.Tensor,
    baseline_scores: t.Tensor,
    unfaithful_labels: t.Tensor,
    *,
    threshold: float = 0.5,
) -> FeatureDetectorReport:
    raise NotImplementedError()


tests.test_feature_detector_report_scores_thresholded_predictions(
    feature_detector_report,
)


## Condition-level Comparison

> ```yaml
> Difficulty: medium
> Importance: medium
> ```

Keep no-CoT, faithful-CoT, biased-CoT, and post-hoc cases separate.

<details>
<summary>Expected output</summary>

```text
All tests in `test_cot_condition_comparison_report_tracks_gaps` passed!
```

The toy fixture has faithful-CoT accuracy `1.0`, biased gap `1/3`, and post-hoc gap `1/3`.
</details>

<details>
<summary>Common bugs</summary>

- Reporting one aggregate accuracy and hiding per-condition failures.
- Allowing missing condition keys.
- Forgetting to reject non-finite condition tensors.
</details>

<details>
<summary>Help - how should I read real condition gaps?</summary>

The real pinned report has `posthoc_gap = -0.5`, so condition behavior is not a simple success story. That is why this section reports condition diagnostics without claiming broad CoT faithfulness.
</details>

<details>
<summary>Solution</summary>

Require exactly the four condition names, average each correctness tensor, then compute `posthoc - biased` and `faithful - posthoc` gaps.
</details>


In [ ]:
def cot_condition_comparison_report(
    condition_correct: dict[CoTCondition, t.Tensor],
) -> CoTConditionComparisonReport:
    raise NotImplementedError()


tests.test_cot_condition_comparison_report_tracks_gaps(
    cot_condition_comparison_report,
)


## Notebook Contract

> ```yaml
> Difficulty: easy
> Importance: high
> ```

Aggregate the toy checks into the smoke-test dictionary used by the report runner.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```
</details>

<details>
<summary>Help - why keep this contract?</summary>

The smoke contract is the fast learner-facing check. The full GT-3 acceptance still comes from the pinned CUDA report, not from toy tensors alone.
</details>

<details>
<summary>Solution</summary>

Return the five nested dictionaries with stable keys: `probe`, `patching`, `text_baseline`, `feature_detector`, and `condition_comparison`.
</details>


In [ ]:
def probe_smoke_test() -> dict:
    probe_logits = t.tensor([[2.0, 0.0], [0.0, 2.0], [2.0, 0.0]])
    hidden_answer_ids = t.tensor([0, 1, 0])
    final_answer_ids = t.tensor([0, 0, 0])
    return pre_final_answer_probe_report(
        probe_logits,
        hidden_answer_ids,
        final_answer_ids,
        min_hidden_accuracy=1.0,
    ).__dict__


def patching_smoke_test() -> dict:
    original_logits = t.tensor([3.0, 0.0])
    patched_logits = t.tensor([0.0, 3.0])
    return hidden_answer_patching_report(original_logits, patched_logits).__dict__


def text_baseline_smoke_test() -> dict:
    labels = t.tensor([1, 0, 1, 0], dtype=t.bool)
    detector = t.tensor([1, 0, 1, 0], dtype=t.bool)
    text_only = t.tensor([0, 0, 1, 0], dtype=t.bool)
    return cot_text_baseline_report(detector, text_only, labels).__dict__


def feature_detector_smoke_test() -> dict:
    labels = t.tensor([1, 0, 1, 0], dtype=t.bool)
    feature_scores = t.tensor([0.9, 0.1, 0.8, 0.2])
    baseline_scores = t.tensor([0.2, 0.1, 0.6, 0.2])
    return feature_detector_report(
        feature_scores,
        baseline_scores,
        labels,
        threshold=0.5,
    ).__dict__


def condition_comparison_smoke_test() -> dict:
    return cot_condition_comparison_report(
        {
            "no_cot": t.tensor([1, 0, 1], dtype=t.float32),
            "faithful_cot": t.tensor([1, 1, 1], dtype=t.float32),
            "biased_cot": t.tensor([1, 0, 0], dtype=t.float32),
            "posthoc": t.tensor([1, 1, 0], dtype=t.float32),
        }
    ).__dict__


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "probe": probe_smoke_test(),
        "patching": patching_smoke_test(),
        "text_baseline": text_baseline_smoke_test(),
        "feature_detector": feature_detector_smoke_test(),
        "condition_comparison": condition_comparison_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)


## Signature Result

<img src="../../instructions/assets/cot_faithfulness_signature_result.svg" width="860">

The committed CUDA report is a narrow Pythia-70M hidden-state A/B private-answer preflight. It records hidden-answer accuracy `1.0`, final-answer agreement `0.5`, label-shuffled accuracy `0.0`, a hidden-state readout flip from answer `0` to `1`, detector recall `1.0` versus text-only recall `0.5`, and peak VRAM about `0.34 GB`.

<details>
<summary>Interpreting the signature result</summary>

The result says the hidden state contains private-answer evidence and that moving the hidden vector changes the LM-head answer readout. It does not say Pythia has faithful natural-language reasoning, and it does not evaluate generated CoT text.
</details>


In [ ]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    tests.test_committed_gpu_report_uses_real_text_only_baseline(gpu)
    assert report["accepted"]
    assert gpu["cuda_available"]
    assert gpu["preflight_passed"]
    assert gpu["model_name"] == "EleutherAI/pythia-70m-deduped"
    assert gpu["hf_revision"] == "e93a9faa9c77e5d09219f6c868bfc7a1bd65593c"
    assert gpu["hidden_answer_accuracy"] == 1.0
    assert gpu["final_answer_agreement"] == 0.5
    assert gpu["label_shuffled_probe_accuracy"] == 0.0
    assert gpu["patching_changed_output"]
    assert gpu["text_only_misses_cases"]
    assert gpu["feature_detector_improves"]
    assert gpu["train_prompt_count"] == 24
    assert gpu["heldout_prompt_count"] == 8
    assert gpu["hidden_state_shape"] == [8, 512]
    assert gpu["unfaithful_case_count"] == 4
    assert gpu["within_vram_budget"]
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test()
utils.print_report(
    "Committed CUDA signature result",
    {
        "device": gpu["device"],
        "hidden_answer_accuracy": gpu["hidden_answer_accuracy"],
        "final_answer_agreement": gpu["final_answer_agreement"],
        "label_shuffled_probe_accuracy": gpu["label_shuffled_probe_accuracy"],
        "patching_changed_output": gpu["patching_changed_output"],
        "detector_vs_text_recall": f'{gpu["detector_recall"]} vs {gpu["text_only_recall"]}',
        "peak_vram_gb": gpu["peak_vram_gb"],
    },
)

tests.test_exercise_notebook_declares_full_verification_contract()


## Limitations

- The toy cells teach the reporting contract; they are not real-model evidence.
- The CUDA path is a Pythia-70M safe A/B hidden-state preflight with eight held-out prompts.
- The patching result is an LM-head readout flip after hidden-vector movement, not a full forward-pass activation patch.
- No chain-of-thought completions are generated or scored.
- The visible-text baseline is intentionally simple; stronger rationale-only baselines are a natural extension.

## Further Research

Try layer/position sweeps, stronger probes, a distribution of label shuffles, a stronger rationale-only baseline, or a true hook-based activation patch through the forward pass.
